# Ops SLA Drilldown — POC

**Design:** `openspec/changes/ops-sla-drilldown/design.md`  
**Queries:** `queries/ops_contribution.sql`, `ops_product_breakdown.sql`, `ops_location_breakdown.sql`, `ops_facility_product.sql`  
**Audience:** Ops team — identify bottleneck trong ≤2 phút  

> **Nguyên tắc:** Notebook này chỉ visualize — không transform, filter, hay tính toán business logic.  
> Toàn bộ SLA logic đã xử lý trong SQL (verified qua MCP).

### Drill-down flow
```
L1 Chart 3 — Facility triage         → HB1/HB2 thấp → escalate
L2 Chart 4 — Facility × product      → production hay scan bottleneck?
L3 Chart 2 — Product breakdown        → worst product priorities
L4 Chart 1 — Contribution split       → confirm bottleneck type per product
```

## 1. Setup

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

REPORT_NAME = 'ops-sla-drilldown'

# SLA thresholds
SLA = {'Platform': 3, 'Standard': 4}

# Colors
COLOR_PROD = '#e67e22'   # orange — production
COLOR_SCAN = '#3498db'   # blue — carrier scan
COLOR_PLT  = '#e67e22'   # Platform
COLOR_STD  = '#3498db'   # Standard
COLOR_TX1  = '#2ecc71'   # benchmark
COLOR_HB1  = '#e74c3c'   # bottleneck
COLOR_HB2  = '#c0392b'   # bottleneck

## 2. Load Data

In [ ]:
df_contrib   = pd.read_csv('queries/ops_contribution.csv')
df_product   = pd.read_csv('queries/ops_product_breakdown.csv')
df_location  = pd.read_csv('queries/ops_location_breakdown.csv')
df_fac_prod  = pd.read_csv('queries/ops_facility_product.csv')

# Loại Unknown facility và data quality outlier (Metal sign first_scan_day âm)
df_location  = df_location[df_location['ordx_fulfill_locate'] != 'Unknown']
df_fac_prod  = df_fac_prod[
    (df_fac_prod['ordx_fulfill_locate'] != 'Unknown') &
    (df_fac_prod['avg_first_scan_day'] >= 0)
]

print(f'Contribution: {len(df_contrib)} rows | Product: {len(df_product)} rows')
print(f'Location: {len(df_location)} rows | Facility×Product: {len(df_fac_prod)} rows')

## 3. Chart 3 — L1: Facility Triage

Pass rate per facility, Platform vs Standard. TX1 = benchmark.

In [ ]:
fig3 = go.Figure()

facility_order = ['HB2', 'HB1', 'TX1']
colors_group = {'Platform': COLOR_PLT, 'Standard': COLOR_STD}

for group in ['Standard', 'Platform']:
    g = df_location[df_location['group'] == group].copy()
    g['ordx_fulfill_locate'] = pd.Categorical(g['ordx_fulfill_locate'], categories=facility_order, ordered=True)
    g = g.sort_values('ordx_fulfill_locate')

    fig3.add_trace(go.Bar(
        name=f'{group} (SLA ≤{SLA[group]}d)',
        x=g['ordx_fulfill_locate'],
        y=g['pass_rate'],
        text=[f"{r:.1f}%<br>n={n:,}" for r, n in zip(g['pass_rate'], g['total_orders'])],
        textposition='outside',
        marker_color=colors_group[group],
        opacity=0.85,
    ))

fig3.update_layout(
    title={
        'text': 'L1 — SLA Pass Rate by Facility<br><sup>6 tháng gần nhất | Platform ≤3d, Standard ≤4d</sup>',
        'font': {'size': 16},
    },
    barmode='group',
    xaxis_title='Facility',
    yaxis={'title': 'Pass Rate (%)', 'range': [0, 115], 'ticksuffix': '%'},
    legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.02},
    height=420,
    paper_bgcolor='white',
    plot_bgcolor='#f8f9fa',
    hovermode='x unified',
)
# TX1 benchmark annotation
fig3.add_annotation(
    x='TX1', y=105,
    text='✓ Benchmark',
    showarrow=False,
    font={'color': COLOR_TX1, 'size': 11},
)
fig3.show()

## 4. Chart 4 — L2: Facility × Product Drill-down

Sau khi L1 identify HB1/HB2 là bottleneck: loại bottleneck nào (production hay scan) tại từng facility?

In [ ]:
# Lọc product có đủ volume (>= 100 orders để tránh outlier nhỏ)
df_fp = df_fac_prod[df_fac_prod['order_count'] >= 100].copy()

facilities = ['HB1', 'HB2', 'TX1']
groups = ['Platform', 'Standard']

fig4 = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f'{g} — {f}' for g in groups for f in facilities],
    shared_yaxes=False,
    vertical_spacing=0.18,
    horizontal_spacing=0.08,
)

shown_legend = set()

for row_idx, group in enumerate(groups, start=1):
    for col_idx, facility in enumerate(facilities, start=1):
        sub = df_fp[(df_fp['group'] == group) & (df_fp['ordx_fulfill_locate'] == facility)].copy()
        if sub.empty:
            continue
        sub = sub.sort_values('avg_tal', ascending=False)

        show_prod = 'prod' not in shown_legend
        show_scan = 'scan' not in shown_legend

        fig4.add_trace(go.Bar(
            name='Production',
            x=sub['product_type'],
            y=sub['avg_processing_day'].round(2),
            marker_color=COLOR_PROD,
            showlegend=show_prod,
            legendgroup='prod',
            hovertemplate='%{x}<br>avg_prod: %{y:.2f}d<extra>Production</extra>',
        ), row=row_idx, col=col_idx)

        fig4.add_trace(go.Bar(
            name='Carrier scan',
            x=sub['product_type'],
            y=sub['avg_first_scan_day'].round(2),
            marker_color=COLOR_SCAN,
            showlegend=show_scan,
            legendgroup='scan',
            hovertemplate='%{x}<br>avg_scan: %{y:.2f}d<extra>Carrier scan</extra>',
        ), row=row_idx, col=col_idx)

        # SLA threshold reference line
        threshold = SLA[group]
        fig4.add_hline(
            y=threshold, line_dash='dash', line_color='red',
            line_width=1.5, opacity=0.6,
            row=row_idx, col=col_idx,
        )

        if show_prod: shown_legend.add('prod')
        if show_scan: shown_legend.add('scan')

fig4.update_layout(
    title={
        'text': 'L2 — Avg Production + Carrier Scan by Facility × Product<br><sup>Stacked bar — dashed line = SLA threshold | Filter: product >= 100 orders</sup>',
        'font': {'size': 15},
    },
    barmode='stack',
    height=680,
    paper_bgcolor='white',
    plot_bgcolor='#f8f9fa',
    legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.04},
)
fig4.show()

## 5. Chart 2 — L3: Product Breakdown (Pass Rate)

Worst → best per group, sorted để ops thấy ngay ưu tiên.

In [ ]:
fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Platform (SLA ≤3d)', 'Standard (SLA ≤4d)'],
    horizontal_spacing=0.12,
)

for col_idx, group in enumerate(['Platform', 'Standard'], start=1):
    g = df_prod[df_prod['group'] == group].sort_values('pass_rate')

    fig2.add_trace(go.Bar(
        x=g['pass_rate'],
        y=g['product_type'],
        orientation='h',
        text=[f"{r:.1f}%  (n={n:,})" for r, n in zip(g['pass_rate'], g['total_orders'])],
        textposition='outside',
        marker_color=colors_group[group],
        opacity=0.85,
        showlegend=False,
        hovertemplate='%{y}: %{x:.1f}%<extra></extra>',
    ), row=1, col=col_idx)

fig2.update_xaxes(range=[0, 120], ticksuffix='%')
fig2.update_layout(
    title={
        'text': 'L3 — SLA Pass Rate by Product Type<br><sup>Sorted worst → best | Filter: >= 100 orders</sup>',
        'font': {'size': 15},
    },
    height=480,
    paper_bgcolor='white',
    plot_bgcolor='#f8f9fa',
)
fig2.show()

## 6. Chart 1 — L4: Contribution Split

Avg production_day vs first_scan_day per product — confirm bottleneck type.

In [ ]:
df_c = df_contrib[df_contrib['order_count'] >= 100].copy()

fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Platform (SLA ≤3d)', 'Standard (SLA ≤4d)'],
    horizontal_spacing=0.12,
)

shown_legend1 = set()

for col_idx, group in enumerate(['Platform', 'Standard'], start=1):
    g = df_c[df_c['group'] == group].sort_values('avg_tal', ascending=False)
    threshold = SLA[group]

    show_prod = 'prod1' not in shown_legend1
    show_scan = 'scan1' not in shown_legend1

    fig1.add_trace(go.Bar(
        name='Production',
        x=g['product_type'],
        y=g['avg_processing_day'].round(2),
        marker_color=COLOR_PROD,
        showlegend=show_prod,
        legendgroup='prod1',
        hovertemplate='%{x}<br>avg_prod: %{y:.2f}d<extra>Production</extra>',
    ), row=1, col=col_idx)

    fig1.add_trace(go.Bar(
        name='Carrier scan',
        x=g['product_type'],
        y=g['avg_first_scan_day'].round(2),
        marker_color=COLOR_SCAN,
        showlegend=show_scan,
        legendgroup='scan1',
        hovertemplate='%{x}<br>avg_scan: %{y:.2f}d<extra>Carrier scan</extra>',
    ), row=1, col=col_idx)

    fig1.add_hline(
        y=threshold, line_dash='dash', line_color='red',
        line_width=1.5, opacity=0.7,
        row=1, col=col_idx,
        annotation_text=f'SLA ≤{threshold}d',
        annotation_position='top right',
    )

    if show_prod: shown_legend1.add('prod1')
    if show_scan: shown_legend1.add('scan1')

fig1.update_layout(
    title={
        'text': 'L4 — Avg Production + Carrier Scan by Product Type (Overall)<br><sup>Sorted by avg TAL desc | Dashed = SLA threshold | Filter: >= 100 orders</sup>',
        'font': {'size': 15},
    },
    barmode='stack',
    yaxis_title='Avg biz days',
    height=460,
    paper_bgcolor='white',
    plot_bgcolor='#f8f9fa',
    legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.04},
)
fig1.show()

## 7. Notes

### Findings
- **L1 (Facility):** HB1 Platform=71.6%, Standard=52% — priority #1 (volume lớn nhất HB). HB2 Platform=62.7%, Standard=44% — priority #2. TX1 healthy (86–89%).
- **L2 (Bottleneck type):** HB production chậm hơn TX1 rõ rệt — Standard HB1 Apparel prod=4.16 vs TX1=1.34. HB2 Platform Tumbler prod=6.07 (anomaly — cần verify).
- **L3 (Product priority):** Standard Tumbler 43.2%, Standard Doormat 60.7%, Standard Poster 62.3% — 3 worst performers.
- **L4 (Bottleneck type global):** Standard Doormat avg_scan=4.34 — carrier issue toàn hệ thống, không phải facility-specific.

### Data quality
- Standard Unknown Metal sign: `first_scan_day = -1.42` (âm) — 1 row, filtered out.
- Platform HB2 Tumbler avg_prod=6.07 với n=946 — cần verify có phải outlier hay seasonal issue.

### Next steps (Metabase)
- 4 queries → 4 cards, layout theo drill-down flow L1→L4.
- Metabase không support stacked+grouped natively — có thể cần tách thành separate cards.